Лабораторная работа №1

Лемматизировать текст (привести к словарной форме) и приписать леммам частеречные теги.
Для решения задачи вы можете использовать данные, которые упоминались в лекциях: например, словарь oDict, разметку OpenCorpora и др. Для решения задачи нельзя использовать существующие морфологические анализаторы (mystem, pymorphy и т.п.).
Ввод: предложения вида " токен1 токен2 ... токенN" с расставленными знаками препинания, разделенные переносом строки. Из знаков препинания в предложениях могут содержаться только запятая, точка, вопросительный и восклицательный знаки.
Вывод: для каждого предложения из входных данных вывод в виде " токен1{лемма1=тег1} токен2{лемма2= тег2} ... токенN{леммаN=тегN}" без исходных знаков препинания. Разделитель между токенами -- пробельный символ.
Замечание: При лемматизации буквы е и ё, а также написание с прописной/строчной буквы признаются равноправными. Не забываем про неизвестные слова и оценку точности.
Пример:
Input:
Стала стабильнее экономическая и политическая обстановка, предприятия вывели из тени зарплаты сотрудников.
Все Гришины одноклассники уже побывали за границей, он был чуть ли не единственным, кого не вывозили никуда дальше Красной Пахры.
Output:
Стала{стать=V} стабильнее{стабильный=A} экономическая{экономический=A} и{и=CONJ} политическая{политический=A} обстановка{обстановка=S} предприятия{предприятие=S} вывели{вывести=V} из{из=PR} тени{тень=S} зарплаты{зарплата=S} сотрудников{сотрудник=S}
Все{весь=NI} Гришины{гришин=A} одноклассники{одноклассник=S} уже{уже=ADV} побывали{побывать=V} за{за=PR} границей{граница=S} он{он=NI} был{быть=V} чуть{чуть=ADV} ли{ли=ADV} не{не=ADV} единственным{единственный=A} кого{кто=NI} не{не=ADV} вывозили{вывозить=V} никуда{никуда=NI} дальше{далеко=ADV} Красной{красный=A} Пахры{Пахра=S}

Был установлен файл с размеченным текстом, содержащий предложения, где для каждого слова представлен морфологический разбор

Пример содержимого файла:

In [1]:
FILE_PATH='/content/annot.opcorpora.xml'

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 50:
            print(line.rstrip())
        else:
            break

<?xml version="1.0" encoding="utf-8" standalone="yes"?>
<annotation version="0.12" revision="4175310">
<text id="1" parent="0" name="&quot;Частный корреспондент&quot;">
  <tags>
    <tag>url:http://www.chaskor.ru</tag>
    <tag>Тип:Газета</tag>
  </tags>
  <paragraphs>
  </paragraphs>
</text>
<text id="2" parent="1" name="00021 Школа злословия">
  <tags>
    <tag>Год:2008</tag>
    <tag>Дата:25/08</tag>
    <tag>Автор:Валентин Колесников</tag>
    <tag>url:http://www.chaskor.ru/article/shkola_zlosloviya_uchit_prikusit_yazyk_21</tag>
    <tag>Тема:ЧасКор:Медиа</tag>
    <tag>Тема:ЧасКор:Медиа/ТВ и радио</tag>
  </tags>
  <paragraphs>
    <paragraph id="1">
      <sentence id="1">
        <source>«Школа злословия» учит прикусить язык</source>
        <tokens>
          <token id="1" text="«"><tfr rev_id="2420236" t="«"><v><l id="0" t="«"><g v="PNCT"/></l></v></tfr></token>
          <token id="2" text="Школа"><tfr rev_id="834910" t="Школа"><v><l id="380220" t="школа"><g v="NOUN"/><g v="i

Замечу, что у инфинитивов глаголов  леммой является инфинитив, а у других форм глаголов лемма это глагол в прошедшем времени, мужского рода

Для того, чтобы убрать это несоответствие, был распаршен словарь dict.opcorpora.txt: каждому слову здесь соответствует корректная лемма, однако здесь используется только словоформа-лемма для глаголов

Логика следующая: в исходном файле слова разбиты по группам: одно слово, разные словоформы
Инфинитив расположен в следующей группе после тех же глаголов в той же форме

In [2]:
def create_morph_dict(filename):
    morph_dict = {}
    groups = []
    current_group = []
    group_id = None

    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.isdigit():
                if current_group:
                    groups.append((group_id, current_group))
                group_id = int(line)
                current_group = []
            else:
                parts = line.split('\t')
                if len(parts) == 2:
                    current_group.append((parts[0], parts[1]))

        if current_group:
            groups.append((group_id, current_group))

    print(f"Найдено групп: {len(groups)}")

    for i, (gid, group) in enumerate(groups):
        has_infinitive = any(tag.startswith('INFN') for _, tag in group)

        if has_infinitive:
            infinitive = None
            for form, tag in group:
                if tag.startswith('INFN'):
                    infinitive = form
                    break

            if infinitive:
                for form, tag in group:
                    pos = tag.split(',')[0].split()[0]
                    key = form.upper()
                    if key not in morph_dict:
                        morph_dict[key] = (infinitive.upper(), pos)

        else:
            if i + 1 < len(groups):
                next_gid, next_group = groups[i + 1]
                next_has_infinitive = any(tag.startswith('INFN') for _, tag in next_group)

                if next_has_infinitive:
                    infinitive = None
                    for form, tag in next_group:
                        if tag.startswith('INFN'):
                            infinitive = form
                            break

                    if infinitive:
                        for form, tag in group:
                            pos = tag.split(',')[0].split()[0]
                            key = form.upper()
                            if key not in morph_dict:
                                morph_dict[key] = (infinitive.upper(), pos)
                        continue

            lemma = find_lemma_in_group(group)
            if lemma:
                for form, tag in group:
                    pos = tag.split(',')[0].split()[0]
                    key = form.upper()
                    if key not in morph_dict:
                        morph_dict[key] = (lemma.upper(), pos)

    print(f"Загружено словоформ: {len(morph_dict)}")
    return morph_dict

def find_lemma_in_group(group):
    for form, tag in group:
        if 'CONJ' in tag or 'PREP' in tag:
            return form

    for form, tag in group:
        if 'NOUN' in tag and 'nomn' in tag and 'sing' in tag:
            return form

    for form, tag in group:
        if 'ADJF' in tag and 'nomn' in tag and 'masc' in tag and 'sing' in tag:
            return form

    for form, tag in group:
        if 'NPRO' in tag and 'nomn' in tag:
            return form

    if group:
        return group[0][0]

    return None

morph_dict = create_morph_dict('dict.opcorpora.txt')

Найдено групп: 391848
Загружено словоформ: 3065338


In [3]:
test_words = ['стала', 'убиться', 'буду', 'рофлил']

print("\nПроверка словаря:")
for word in test_words:
    key = word.upper()
    if key in morph_dict:
        lemma, pos = morph_dict[key]
        print(f"{word:15} -> {lemma:15} = {pos}")
    else:
        print(f"{word:15} -> oov")


Проверка словаря:
стала           -> СТАТЬ           = VERB
убиться         -> УБИТЬСЯ         = INFN
буду            -> БЫТЬ            = VERB
рофлил          -> oov


Парсинг и маппинг корпуса

XML файл приводится к виду List[List[Tuple[str, str, str]]], текст разбивается на список предложений, внутри которых хранится набор из слоформы, леммы и POS

Если встречается словоформа-глагол, ее лемма определяется через morph_dict: отпаршенном словаре dict.opcorpora.txt.

In [4]:
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from lxml import etree
import torch
import torch.nn as nn
import re

def parse_annotated_corpus(xml_path, max_sentences=None, morph_dict=None):
    """
    morph_dict: словарь из create_morph_dict(),
                формат: {WORD_FORM_UPPER: (LEMMA_UPPER, POS)}
    """
    tag_map = {
        'NOUN': 'S', 'VERB': 'V', 'ADJF': 'A', 'ADJS': 'A',
        'PTCP': 'A', 'PRTF': 'A', 'PRTS': 'A', 'COMP': 'A',
        'ADV': 'ADV', 'ADVB': 'ADV', 'GRND': 'ADV',
        'PREP': 'PR', 'CONJ': 'CONJ',
        'PRON': 'NI', 'NPRO': 'NI', 'NUMR': 'NI',
        'PRED': 'PRED', 'INTJ': 'INTJ', 'INFN': 'V',
        'PNCT': 'PNCT'
    }

    sentences = []
    current_sentence = []
    sentence_count = 0

    context = etree.iterparse(xml_path, events=('end',), tag='token', recover=True)

    try:
        for event, elem in context:
            word = elem.get('text')
            if not word:
                elem.clear()
                continue

            lemma_elem = elem.find('.//l')
            if lemma_elem is not None:
                lemma = lemma_elem.get('t')

                pos_raw = None
                for gram in lemma_elem.findall('g'):
                    val = gram.get('v')
                    if val and val.isupper() and 3 <= len(val) <= 6:
                        pos_raw = val
                        break

                if lemma and pos_raw:
                    pos_tag = tag_map.get(pos_raw, 'S')

                    if pos_tag == 'V' and morph_dict:
                        word_key = word.upper()
                        if word_key in morph_dict:
                            final_lemma, _ = morph_dict[word_key]
                            final_lemma = final_lemma.lower()
                        else:
                            final_lemma = lemma
                    else:
                        final_lemma = lemma

                    current_sentence.append((word, final_lemma, pos_tag))

            if word in ['.', '!', '?', '…', '...']:
                if len(current_sentence) >= 3:
                    sentences.append(current_sentence)
                    sentence_count += 1
                current_sentence = []

            elem.clear()
            while elem.getprevious() is not None:
                del elem.getparent()[0]

            if max_sentences and sentence_count >= max_sentences:
                break

    except Exception as e:
        print(f"Ошибка парсинга: {e}")

    if current_sentence and len(current_sentence) >= 3:
        sentences.append(current_sentence)

    print(f"Извлечено предложений: {len(sentences):,}")
    return sentences

In [5]:
result = parse_annotated_corpus('/content/annot.opcorpora.xml', max_sentences=2, morph_dict=morph_dict)

print("\nПримеры распарсенных данных:")
for i, sent in enumerate(result, 1):
    print(f"\nПредложение {i}:")
    for word, lemma, pos in sent:
        print(f"  {word:15} : {lemma:15} = {pos} ")

Извлечено предложений: 2

Примеры распарсенных данных:

Предложение 1:
  «               : «               = PNCT 
  Школа           : школа           = S 
  злословия       : злословие       = S 
  »               : »               = PNCT 
  учит            : учить           = V 
  прикусить       : прикусить       = V 
  язык            : язык            = S 
  Сохранится      : сохраниться     = V 
  ли              : ли              = S 
  градус          : градус          = S 
  дискуссии       : дискуссия       = S 
  в               : в               = PR 
  новом           : новый           = A 
  сезоне          : сезон           = S 
  ?               : ?               = PNCT 

Предложение 2:
  Великолепная    : великолепный    = A 
  «               : «               = PNCT 
  Школа           : школа           = S 
  злословия       : злословие       = S 
  »               : »               = PNCT 
  вернулась       : вернуться       = V 
  в               : в               

Подкотовка словаря для обучения модели

char2id - перевод текст в числа для модели

tag2id - перевод POS-тегов в числа для функции потерь

id2tag - перевод предсказаний модели обратно в читаемые теги

In [6]:
def build_vocabularies():
    char_vocab = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'

    char2id = {char: idx + 2 for idx, char in enumerate(char_vocab)}

    char2id['<PAD>'] = 0
    char2id['<UNK>'] = 1
    char2id['<BOS>'] = 35
    char2id['<EOS>'] = 36

    all_tags = ['<PAD>', '<UNK>', 'S', 'V', 'A', 'ADV', 'PR', 'CONJ', 'NI', 'PRED', 'INTJ', 'PNCT']

    tag2id = {tag: idx for idx, tag in enumerate(all_tags)}
    id2tag = {idx: tag for tag, idx in tag2id.items()}

    return char2id, tag2id, id2tag

Класс ContextualMorphDataset наследуется предназначен для подготовки морфологического корпуса к обучению модели. Он преобразует 'сырые' предложения (список кортежей слово-лемма-POS) в тензоры фиксированной размерности. Выполняет нормализацию текста (нижний регистр, замена ё на е), символьное кодирование через словарь char2id, добавление специальных токенов EOS для лемм и паддинг до max_word_len и max_sent_len. Возвращает словарь с ключами input_chars, target_pos, target_lemma и length.

In [7]:
class ContextualMorphDataset(Dataset):
    def __init__(
        self,
        sentences,
        char2id,
        tag2id,
        max_word_len=30,
        max_sent_len=50
    ):
        self.sentences = sentences
        self.char2id = char2id
        self.tag2id = tag2id
        self.max_word_len = max_word_len
        self.max_sent_len = max_sent_len

    def __len__(self):
        return len(self.sentences)

    def _encode_word(self, word, is_lemma=False):
        word = word.lower().replace('ё', 'е')

        limit = self.max_word_len - 1 if is_lemma else self.max_word_len
        ids = [self.char2id.get(char, 1) for char in word[:limit]]

        if is_lemma:
            ids.append(self.char2id['<EOS>'])

        ids += [0] * (self.max_word_len - len(ids))
        return ids

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        n_words = min(len(sentence), self.max_sent_len)

        input_chars = []
        target_pos = []
        target_lemma = []

        for i in range(self.max_sent_len):
            if i < n_words:
                word, lemma, pos = sentence[i]

                input_chars.append(self._encode_word(word, is_lemma=False))
                target_pos.append(self.tag2id.get(pos, 0))

                target_lemma.append(self._encode_word(lemma, is_lemma=True))
            else:
                input_chars.append([0] * self.max_word_len)
                target_pos.append(0)
                target_lemma.append([0] * self.max_word_len)

        return {
            'input_chars': torch.tensor(input_chars, dtype=torch.long),
            'target_pos': torch.tensor(target_pos, dtype=torch.long),
            'target_lemma': torch.tensor(target_lemma, dtype=torch.long),
            'length': n_words
        }

In [8]:
char2id, tag2id, id2tag = build_vocabularies()

print("Словари:")
print(f"  Размер char2id: {len(char2id)} символов")
print(f"  Размер tag2id: {len(tag2id)} тегов")
print(f"  tag2id: {tag2id}")
print(f"  id2tag: {id2tag}\n")

print("Парсим 5 предложений для теста...")
sentences = parse_annotated_corpus('/content/annot.opcorpora.xml', max_sentences=5)

dataset = ContextualMorphDataset(
    sentences, char2id, tag2id,
    max_word_len=15,
    max_sent_len=10
)

print(f"Датасет создан: {len(dataset)} предложений\n")

print("Пример кодирования первого предложения:\n")
sample = dataset[0]

print(f"Структура образца:")
print(f"  input_chars shape: {sample['input_chars'].shape}")
print(f"  target_pos shape:  {sample['target_pos'].shape}")
print(f"  target_lemma shape: {sample['target_lemma'].shape}")
print(f"  length: {sample['length']}\n")

print("Первые 5 токенов (оригинал → код → декод):")
print(f"{'Слово':<15} {'Лемма':<15} {'POS':<6} {'Word IDs (первые 10)':<30} {'Lemma IDs (первые 10)':<30}")
print("-" * 100)

original_sent = sentences[0]
for i in range(min(5, len(original_sent))):
    word, lemma, pos = original_sent[i]

    word_ids = sample['input_chars'][i].tolist()
    lemma_ids = sample['target_lemma'][i].tolist()
    pos_id = sample['target_pos'][i].item()

    word_ids_short = word_ids[:10]
    lemma_ids_short = lemma_ids[:10]

    print(f"{word:<15} {lemma:<15} {pos:<6} {str(word_ids_short):<30} {str(lemma_ids_short):<30}")

Словари:
  Размер char2id: 37 символов
  Размер tag2id: 12 тегов
  tag2id: {'<PAD>': 0, '<UNK>': 1, 'S': 2, 'V': 3, 'A': 4, 'ADV': 5, 'PR': 6, 'CONJ': 7, 'NI': 8, 'PRED': 9, 'INTJ': 10, 'PNCT': 11}
  id2tag: {0: '<PAD>', 1: '<UNK>', 2: 'S', 3: 'V', 4: 'A', 5: 'ADV', 6: 'PR', 7: 'CONJ', 8: 'NI', 9: 'PRED', 10: 'INTJ', 11: 'PNCT'}

Парсим 5 предложений для теста...
Извлечено предложений: 5
Датасет создан: 5 предложений

Пример кодирования первого предложения:

Структура образца:
  input_chars shape: torch.Size([10, 15])
  target_pos shape:  torch.Size([10])
  target_lemma shape: torch.Size([10, 15])
  length: 10

Первые 5 токенов (оригинал → код → декод):
Слово           Лемма           POS    Word IDs (первые 10)           Lemma IDs (первые 10)         
----------------------------------------------------------------------------------------------------
«               «               PNCT   [1, 0, 0, 0, 0, 0, 0, 0, 0, 0] [1, 36, 0, 0, 0, 0, 0, 0, 0, 0]
Школа           школа           S 

Модель MorphModel предназначена для определения части речи и генерации леммы для каждого слова во входной последовательности.На вход подаётся тензор индексов символов [batch_size, seq_len, word_len].

Сначала каждое слово кодируется независимо: символы преобразуются в эмбеддинги и проходят через двунаправленный LSTM (char_encoder), выдающий вектор фиксированной размерности hidden_dim. Затем последовательность словых представлений обрабатывается вторым двунаправленным LSTM (context_lstm), учитывающим контекст предложения.

Полученные контекстные векторы используются двумя выходными слоями: линейным слоем для классификации части речи и декодером лемм, который пошагово генерирует символы целевой леммы. На каждом шаге вход декодера — эмбеддинг предыдущего символа, конкатенированный с контекстным вектором слова. При обучении применяется teacher forcing: с вероятностью teacher_forcing_ratio на вход подаётся правильный символ из целевой последовательности, иначе используется символ предсказанный моделью.

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MorphModel(nn.Module):
    def __init__(self, char_vocab_size, char_emb_dim=128, hidden_dim=512,
                 n_pos_tags=10, max_word_len=30):
        super().__init__()

        self.char_vocab_size = char_vocab_size
        self.char_emb_dim = char_emb_dim
        self.hidden_dim = hidden_dim
        self.n_pos_tags = n_pos_tags
        self.max_word_len = max_word_len
        self.pad_id = 0
        self.unk_id = 1
        self.bos_id = 35
        self.eos_id = 36

        self.char_embedding = nn.Embedding(
            char_vocab_size, char_emb_dim, padding_idx=0
        )

        self.char_encoder = nn.LSTM(
            char_emb_dim, hidden_dim // 2,
            bidirectional=True,
            batch_first=True,
            num_layers=1
        )

        self.context_lstm = nn.LSTM(
            hidden_dim, hidden_dim // 2,
            bidirectional=True,
            batch_first=True,
            num_layers=1
        )

        self.pos_head = nn.Linear(hidden_dim, n_pos_tags)

        self.lemma_decoder = nn.LSTM(
            hidden_dim + char_emb_dim, hidden_dim,
            batch_first=True,
            num_layers=1
        )
        self.lemma_fc = nn.Linear(hidden_dim, char_vocab_size)

        self.dropout = nn.Dropout(0.3)

    def encode_word(self, char_ids):
        batch_size, seq_len, word_len = char_ids.shape

        char_emb = self.char_embedding(char_ids)
        char_emb = char_emb.view(batch_size * seq_len, word_len, -1)

        _, (char_hidden, _) = self.char_encoder(char_emb)
        char_hidden = char_hidden.transpose(0, 1).contiguous()
        char_hidden = char_hidden.view(batch_size, seq_len, -1)

        return char_hidden

    def forward(self, char_ids, lemma_ids=None, teacher_forcing_ratio=0.5):
      batch_size, seq_len, _ = char_ids.shape

      word_repr = self.encode_word(char_ids)
      word_repr = self.dropout(word_repr)

      context_out, _ = self.context_lstm(word_repr)
      context_out = self.dropout(context_out)

      pos_logits = self.pos_head(context_out)

      lemma_logits = torch.zeros(
          batch_size, seq_len, self.max_word_len, self.char_vocab_size,
          device=char_ids.device
      )

      for i in range(seq_len):
          word_context = context_out[:, i, :]

          decoder_hidden = (
              word_context.unsqueeze(0),
              word_context.unsqueeze(0)
          )

          decoder_input = torch.full(
              (batch_size, 1), self.bos_id,
              dtype=torch.long, device=char_ids.device
          )

          for t in range(self.max_word_len):
              char_emb = self.char_embedding(decoder_input)

              decoder_input_with_context = torch.cat(
                  [char_emb, word_context.unsqueeze(1)], dim=2
              )

              decoder_out, decoder_hidden = self.lemma_decoder(
                  decoder_input_with_context, decoder_hidden
              )

              char_logits = self.lemma_fc(decoder_out.squeeze(1))
              lemma_logits[:, i, t, :] = char_logits

              if lemma_ids is not None and torch.rand(1).item() < teacher_forcing_ratio:
                  next_char = lemma_ids[:, i, t:t+1]
              else:
                  next_char = torch.argmax(char_logits, dim=1, keepdim=True)

              decoder_input = next_char

      return pos_logits, lemma_logits

Функция evaluate оценивает качество модели на валидационных или тестовых данных. Она переключает модель в режим оценки, отключает вычисление градиентов и teacher_forcing. Для каждого батча вычисляются потеря и точность для задач POS-тегирования и лемматизации. В конце возвращаются средние потери и точность в процентах для обеих задач.

In [10]:
def evaluate(model, loader, device, criterion_pos, criterion_lemma):
    model.eval()
    total_loss_pos, total_loss_lemma = 0.0, 0.0
    correct_pos, total_pos = 0, 0
    correct_lemma, total_lemma = 0, 0

    with torch.no_grad():
        for batch in loader:
            char_ids = batch['input_chars'].to(device)
            pos_ids = batch['target_pos'].to(device)
            lemma_ids = batch['target_lemma'].to(device)

            pos_logits, lemma_logits = model(char_ids, lemma_ids, teacher_forcing_ratio=0.0)

            b, s, n_tags = pos_logits.shape
            loss_pos = criterion_pos(pos_logits.view(-1, n_tags), pos_ids.view(-1))

            b, s, max_len, vocab = lemma_logits.shape
            loss_lemma = criterion_lemma(lemma_logits.view(-1, vocab), lemma_ids.view(-1))

            total_loss_pos += loss_pos.item()
            total_loss_lemma += loss_lemma.item()

            _, predicted = torch.max(pos_logits, 2)
            mask = pos_ids != 0
            correct_pos += (predicted == pos_ids)[mask].sum().item()
            total_pos += mask.sum().item()

            _, lemma_predicted = torch.max(lemma_logits, 3)
            lemma_mask = lemma_ids != 0
            correct_lemma += (lemma_predicted == lemma_ids)[lemma_mask].sum().item()
            total_lemma += lemma_mask.sum().item()

    acc_pos = correct_pos / total_pos * 100 if total_pos > 0 else 0
    acc_lemma = correct_lemma / total_lemma * 100 if total_lemma > 0 else 0
    loss_pos = total_loss_pos / len(loader)
    loss_lemma = total_loss_lemma / len(loader)

    return loss_pos, loss_lemma, acc_pos, acc_lemma

Функция train_model организует полный цикл обучения модели. Инициализируются функции потерь CrossEntropyLoss с игнорированием паддинга, оптимизатор Adam и планировщик скорости обучения ReduceLROnPlateau.

На каждой эпохе модель проходит тренировочный датасет: вычисляются потери для POS-тегирования и лемматизации, выполняется обновление весов. После каждой эпохи проводится валидация через функцию evaluate, метрики сохраняются в историю.

После завершения всех эпох выполняется финальная оценка на тестовом датасете.

In [11]:
def train_model(model, train_loader, val_loader, test_loader, device, epochs=10, lr=0.0005, teacher_forcing_ratio=0.7):
    criterion_pos = nn.CrossEntropyLoss(ignore_index=0)
    vocab_size = model.char_vocab_size
    token_weights = torch.ones(vocab_size, device=device)
    criterion_lemma = nn.CrossEntropyLoss(ignore_index=0, weight=token_weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    model.to(device)

    history = {
        'train_loss_pos': [], 'train_loss_lemma': [],
        'train_acc_pos': [], 'train_acc_lemma': [],
        'val_loss_pos': [], 'val_loss_lemma': [],
        'val_acc_pos': [], 'val_acc_lemma': []
    }

    for epoch in range(epochs):
        model.train()
        total_loss_pos, total_loss_lemma = 0.0, 0.0
        correct_pos, total_pos = 0, 0
        correct_lemma, total_lemma = 0, 0

        for batch in train_loader:
            char_ids = batch['input_chars'].to(device)
            pos_ids = batch['target_pos'].to(device)
            lemma_ids = batch['target_lemma'].to(device)

            optimizer.zero_grad()
            pos_logits, lemma_logits = model(char_ids, lemma_ids, teacher_forcing_ratio=teacher_forcing_ratio)

            b, s, n_tags = pos_logits.shape
            loss_pos = criterion_pos(pos_logits.view(-1, n_tags), pos_ids.view(-1))
            b, s, max_len, vocab = lemma_logits.shape
            loss_lemma = criterion_lemma(lemma_logits.view(-1, vocab), lemma_ids.view(-1))

            loss = loss_pos + loss_lemma
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss_pos += loss_pos.item()
            total_loss_lemma += loss_lemma.item()

            _, predicted = torch.max(pos_logits, 2)
            mask = pos_ids != 0
            correct_pos += (predicted == pos_ids)[mask].sum().item()
            total_pos += mask.sum().item()

            _, lemma_predicted = torch.max(lemma_logits, 3)
            lemma_mask = lemma_ids != 0
            correct_lemma += (lemma_predicted == lemma_ids)[lemma_mask].sum().item()
            total_lemma += lemma_mask.sum().item()

        train_loss_pos = total_loss_pos / len(train_loader)
        train_loss_lemma = total_loss_lemma / len(train_loader)
        train_acc_pos = correct_pos / total_pos * 100 if total_pos > 0 else 0
        train_acc_lemma = correct_lemma / total_lemma * 100 if total_lemma > 0 else 0

        val_loss_pos, val_loss_lemma, val_acc_pos, val_acc_lemma = evaluate(
            model, val_loader, device, criterion_pos, criterion_lemma
        )

        scheduler.step(val_loss_lemma)

        history['train_loss_pos'].append(train_loss_pos)
        history['train_loss_lemma'].append(train_loss_lemma)
        history['train_acc_pos'].append(train_acc_pos)
        history['train_acc_lemma'].append(train_acc_lemma)
        history['val_loss_pos'].append(val_loss_pos)
        history['val_loss_lemma'].append(val_loss_lemma)
        history['val_acc_pos'].append(val_acc_pos)
        history['val_acc_lemma'].append(val_acc_lemma)

        print(f"Epoch {epoch + 1}/{epochs} | Train POS: {train_acc_pos:.2f}% | Val POS: {val_acc_pos:.2f}% | "
              f"Train Lemma: {train_acc_lemma:.2f}% | Val Lemma: {val_acc_lemma:.2f}%")

    print("\nИтоговая оценка")
    test_loss_pos, test_loss_lemma, test_acc_pos, test_acc_lemma = evaluate(
        model, test_loader, device, criterion_pos, criterion_lemma
    )
    print(f"Test POS Accuracy: {test_acc_pos:.2f}%")
    print(f"Test Lemma Accuracy: {test_acc_lemma:.2f}%")

    return model, history

Функция интерфейса predict_sentence принимает текстовое предложение, выполняет токенизацию и кодирование символов в числовые ID. Данные передаются в модель для прямого прохода (кодирование, контекстуализация, POS-классификация, генерация лемм). Результат декодируется из ID обратно в символы и POS-теги, возвращаясь в формате Словоформа{лемма=POS}.

In [12]:
def predict_sentence(sentence, model, char2id, id2tag, device, max_word_len=30, max_sent_len=20):
    model.eval()

    words = re.findall(r'[а-яА-ЯёЁ]+', sentence)
    n_words = len(words)

    if n_words == 0:
        return []

    char_ids = []
    for word in words:
        w_chars = [char2id.get(char, 1) for char in word.lower().replace('ё', 'е')[:max_word_len]]
        w_chars += [0] * (max_word_len - len(w_chars))
        char_ids.append(w_chars)

    for _ in range(max_sent_len - n_words):
        char_ids.append([0] * max_word_len)

    char_tensor = torch.tensor([char_ids], dtype=torch.long, device=device)

    BOS_ID = 35
    EOS_ID = 36

    with torch.no_grad():
        word_repr = model.encode_word(char_tensor)
        context_out, _ = model.context_lstm(word_repr)
        pos_logits = model.pos_head(context_out)
        pos_preds = torch.argmax(pos_logits[0][:n_words], dim=1).cpu().numpy()

        lemma_preds = []
        id2char = {idx: char for char, idx in char2id.items()}

        for i in range(n_words):
            word_lemma_chars = []
            word_context = context_out[0, i, :].unsqueeze(0).unsqueeze(0)

            decoder_hidden = (word_context, word_context)

            decoder_input = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)

            recent_chars = []
            original_len = len(words[i])

            for t in range(max_word_len):
                char_emb = model.char_embedding(decoder_input)
                decoder_input_with_context = torch.cat([char_emb, word_context], dim=2)

                decoder_out, decoder_hidden = model.lemma_decoder(
                    decoder_input_with_context, decoder_hidden
                )

                char_logits = model.lemma_fc(decoder_out.squeeze(1))
                char_pred = torch.argmax(char_logits, dim=1).item()

                if char_pred == EOS_ID:
                    break

                char = id2char.get(char_pred, '')
                recent_chars.append(char)
                if len(recent_chars) >= 4:
                    last_2 = ''.join(recent_chars[-2:])
                    if last_2 * 2 == ''.join(recent_chars[-4:]):
                        break
                if len(recent_chars) >= 3:
                    if recent_chars[-1] == recent_chars[-2] == recent_chars[-3]:
                        break
                if len(word_lemma_chars) >= original_len + 3:
                    break

                word_lemma_chars.append(char)
                decoder_input = torch.tensor([[char_pred]], dtype=torch.long, device=device)

            lemma_preds.append(''.join(word_lemma_chars))

    result = []
    for i, word in enumerate(words):
        pos = id2tag.get(pos_preds[i], 'S')
        lemma = lemma_preds[i] if lemma_preds[i] else words[i].lower()
        result.append(f"{word}{{{lemma}={pos}}}")

    return result

Сохранение и загрузка модели

In [13]:
import os

def save_checkpoint(model, char2id, tag2id, id2tag, config, path):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'char2id': char2id,
        'tag2id': tag2id,
        'id2tag': id2tag,
        'config': config
    }
    torch.save(checkpoint, path)
    print(f"Чекпоинт сохранен {path}")


def load_checkpoint(path, device='cpu'):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Чекпоинт не найден: {path}")

    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint['config']

    model = MorphModel(
        char_vocab_size=len(checkpoint['char2id']),
        char_emb_dim=config['char_emb_dim'],
        hidden_dim=config['hidden_dim'],
        n_pos_tags=len(checkpoint['tag2id']),
        max_word_len=config['max_word_len']
    )

    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    return (
        model,
        checkpoint['char2id'],
        checkpoint['tag2id'],
        checkpoint['id2tag'],
        config
    )

In [14]:
def main():
    XML_PATH = '/content/annot.opcorpora.xml'
    MAX_SENTENCES = 50000
    MAX_WORD_LEN = 30
    MAX_SENT_LEN = 20
    BATCH_SIZE = 64
    EPOCHS = 5
    LEARNING_RATE = 0.0001
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    char2id, tag2id, id2tag = build_vocabularies()
    print("Парсинг корпуса...")
    data = parse_annotated_corpus(XML_PATH, MAX_SENTENCES, morph_dict=morph_dict)

    dataset = ContextualMorphDataset(
        data, char2id, tag2id,
        max_word_len=MAX_WORD_LEN, max_sent_len=MAX_SENT_LEN
    )

    total_size = len(dataset)
    train_size = int(0.7 * total_size)
    val_size = int(0.15 * total_size)
    test_size = int(0.15 * total_size)

    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size], generator=generator
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = MorphModel(
        char_vocab_size=len(char2id),
        char_emb_dim=128,
        hidden_dim=512,
        n_pos_tags=len(tag2id),
        max_word_len=MAX_WORD_LEN
    )

    print(f"Параметры модели: {sum(p.numel() for p in model.parameters()):,}")
    print("\nНачало обучения")

    model, history = train_model(
        model, train_loader, val_loader, test_loader, DEVICE,
        epochs=EPOCHS, lr=LEARNING_RATE, teacher_forcing_ratio=0.7
    )

    config = {
        'char_emb_dim': 128, 'hidden_dim': 512,
        'max_word_len': MAX_WORD_LEN, 'n_pos_tags': len(tag2id)
    }
    save_checkpoint(model, char2id, tag2id, id2tag, config, '/content/morph_checkpoint.pth')

if __name__ == '__main__':
    main()

Парсинг корпуса...
Извлечено предложений: 50,000
Параметры модели: 4,760,753

Начало обучения
Epoch 1/5 | Train POS: 69.79% | Val POS: 85.36% | Train Lemma: 40.08% | Val Lemma: 42.44%
Epoch 2/5 | Train POS: 87.31% | Val POS: 89.56% | Train Lemma: 61.49% | Val Lemma: 56.35%
Epoch 3/5 | Train POS: 90.38% | Val POS: 92.16% | Train Lemma: 71.98% | Val Lemma: 68.55%
Epoch 4/5 | Train POS: 92.32% | Val POS: 93.48% | Train Lemma: 78.63% | Val Lemma: 76.88%
Epoch 5/5 | Train POS: 93.62% | Val POS: 94.32% | Train Lemma: 83.00% | Val Lemma: 81.89%

Итоговая оценка
Test POS Accuracy: 94.47%
Test Lemma Accuracy: 81.48%
Чекпоинт сохранен /content/morph_checkpoint.pth


Тест

In [15]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, char2id, tag2id, id2tag, config = load_checkpoint('/content/morph_checkpoint.pth', DEVICE)

Пример со стандартным предложением

In [16]:
test_sentence = "Сохранится ли градус дискуссии в новом сезоне?"
print(f"\nInput:  {test_sentence}")
result = predict_sentence(test_sentence, model, char2id, id2tag, DEVICE, config['max_word_len'])
print(f"Output: {' '.join(result)}")


Input:  Сохранится ли градус дискуссии в новом сезоне?
Output: Сохранится{сохраниться=V} ли{ли=CONJ} градус{градст=S} дискуссии{дисскус=S} в{в=PR} новом{ново=S} сезоне{сезно=S}


Пример с разными времанами для глаголов и разными падежами для существительных

In [17]:
test_sentence = "Я писал письмо вчера а завтра буду писать ещё, пока не дождусь ответного письма"
print(f"\nInput:  {test_sentence}")
result = predict_sentence(test_sentence, model, char2id, id2tag, DEVICE, config['max_word_len'])
print(f"Output: {' '.join(result)}")


Input:  Я писал письмо вчера а завтра буду писать ещё, пока не дождусь ответного письма
Output: Я{я=NI} писал{писать=V} письмо{письмо=ADV} вчера{вчер=S} а{а=CONJ} завтра{завтр=S} буду{буд=S} писать{писать=V} ещё{еще=S} пока{пока=ADV} не{не=S} дождусь{дождусь=V} ответного{ответный=A} письма{письма=S}


Пример с наречиями и прдикатами

In [18]:
test_sentence = "Очень жаль что нельзя прийти рано утром"
print(f"\nInput:  {test_sentence}")
result = predict_sentence(test_sentence, model, char2id, id2tag, DEVICE, config['max_word_len'])
print(f"Output: {' '.join(result)}")


Input:  Очень жаль что нельзя прийти рано утром
Output: Очень{очень=ADV} жаль{жаль=S} что{что=CONJ} нельзя{нельзя=S} прийти{прити=V} рано{рано=S} утром{утро=S}


Пример со словами неопределенными в словаре

In [19]:
test_sentence = "Я загуглил этот мем и начал рофлить с друзьями"
print(f"\nInput:  {test_sentence}")
result = predict_sentence(test_sentence, model, char2id, id2tag, DEVICE, config['max_word_len'])
print(f"Output: {' '.join(result)}")


Input:  Я загуглил этот мем и начал рофлить с друзьями
Output: Я{я=NI} загуглил{загуглять=V} этот{этот=A} мем{ме=NI} и{и=CONJ} начал{начал=V} рофлить{ролить=V} с{с=PR} друзьями{друмя=S}


Пример с омонимами. В примере приведены слова в одинаковыми корнями для того, чтобы оценить как модель определяет леммы и части речи.

In [20]:
test_sentence = "Желтый уж уже приполз. Косая подруга косила траву с динной косой."
print(f"\nInput:  {test_sentence}")
result = predict_sentence(test_sentence, model, char2id, id2tag, DEVICE, config['max_word_len'])
print(f"Output: {' '.join(result)}")


Input:  Желтый уж уже приполз. Косая подруга косила траву с динной косой.
Output: Желтый{желой=A} уж{уж=S} уже{уж=S} приполз{припоз=S} Косая{косай=A} подруга{подруг=S} косила{косил=S} траву{трав=S} с{с=PR} динной{динный=S} косой{кос=S}
